# Vector Store vs Vector Database

In the previous section we turned text into **embeddings**: lists of numbers that capture meaning. A real RAG system can produce thousands or millions of them, which raises the next question: **where do we keep them, and how do we search them quickly?**

You'll hear two answers again and again:

- **Vector store**
- **Vector database**

People often use the two terms interchangeably, but they don't mean the same thing. This notebook explains the difference so you can pick the right tool for your project.

> **In one sentence:** a **vector store** is a place to *keep and search* vectors. A **vector database** is a full *database system* built around vectors: it stores and searches them too, and it also handles persistence, updates, scaling, security and many users at once.

## What actually gets stored?

Before comparing the two, it helps to see what a single record looks like. A vector store and a vector database both hold entries with the same four parts:

```
┌──────────────────────────────────────────────────────────────┐
│ id        : "chunk_042"                                      │
│ vector    : [0.12, -0.48, 0.91, ..., 0.05]  ← the embedding  │
│ text      : "Transformers rely on self-attention to ..."     │
│ metadata  : {"source": "1706.03762v7.pdf", "page": 3}        │
└──────────────────────────────────────────────────────────────┘
```

- **id**: a unique key, so the record can be found, updated or deleted.
- **vector**: the embedding. Similarity search compares these.
- **text**: the original chunk. This is what we eventually send to the LLM.
- **metadata**: extra fields (source file, page, date, author, ...) used for filtering and citations.

The search works the same way in both: embed the user's question, then find the stored vectors closest to it (for example with cosine similarity, as in section 02). The difference lies in **everything around that search**.

## What is a Vector Store?

A **vector store** is a component with one main job:

1. **store** embeddings (plus their text and metadata), and
2. **return the most similar ones** for a query vector.

It is usually a **library** that runs **inside your Python process**, keeps data **in memory**, and can optionally save it to a local file or folder.

```
┌──────────────── your Python program ─────────────────┐
│                                                      │
│   your code ──► vector store (in memory)             │
│                   ├─ vectors + text + metadata       │
│                   └─ similarity search               │
│                            │                         │
│                            ▼                         │
│                 optional: save to a local file       │
└──────────────────────────────────────────────────────┘
```

**Typical characteristics**

- **Lightweight:** `pip install` and go, no server to run
- **Fast**, but limited to what **one machine's memory** can hold
- **Few database features:** updates, deletes and filtering are basic (it varies by library), and there are no user accounts, replication or backups
- **Not durable by default:** when the program stops, in-memory data is gone unless you explicitly save it

**Examples**

| Tool | Notes |
|---|---|
| **FAISS** (Meta) | Highly optimised similarity-search library. Stores only the vectors; text and metadata are kept by a wrapper such as LangChain |
| **Chroma** (embedded mode) | Runs inside your process and saves to a local folder |
| **LangChain `InMemoryVectorStore`** | Keeps everything in a Python dictionary; handy for demos and tests |
| **Annoy, hnswlib, ScaNN** | Approximate nearest-neighbour search libraries |

**Analogy:** a vector store is like a **personal bookshelf**. It's right next to you, grabbing a book is quick, and it's perfect for your own collection. But it isn't built for thousands of visitors, there's no librarian, and if the room floods you lose everything.

## What is a Vector Database?

A **vector database** is a complete **database management system** that treats vectors as a first-class data type. It does everything a vector store does, and adds the features you expect from a "real" database:

- **Persistence:** data is written durably to disk and survives restarts and crashes
- **CRUD:** create, read, update and delete individual records at any time, and the changes are searchable almost immediately
- **Metadata filtering:** combine similarity with conditions, e.g. *"the most similar chunks **where** `source = "hr_policy.pdf"` **and** `year >= 2024`"*
- **Scalability:** split (shard) the data across many machines to handle billions of vectors
- **High availability:** replication, backups and recovery
- **Concurrency:** many users and applications reading and writing at the same time
- **Security:** authentication, access control, multi-tenancy, encryption
- **Hybrid search** (in many of them): mix vector similarity with keyword (BM25) search
- **An API:** it runs as a separate service you connect to over the network (REST / gRPC), or as a managed cloud service

```
 app 1 ──┐                         ┌──────── vector database server ────────┐
 app 2 ──┼─► network (REST/gRPC) ─►│ authentication & access control        │
 app 3 ──┘                         │ query engine + metadata filters        │
                                   │ ANN indexes (HNSW, IVF, ...)           │
                                   │ durable storage, replicas, backups     │
                                   │ shards across nodes: node1 node2 ...   │
                                   └────────────────────────────────────────┘
```

**Examples**

| Tool | Notes |
|---|---|
| **Pinecone** | Fully managed, serverless cloud service |
| **Weaviate** | Open source; strong hybrid (vector + keyword) search |
| **Qdrant** | Open source (written in Rust); rich filtering; self-host or cloud |
| **Milvus** / Zilliz Cloud | Open source and distributed, designed for billions of vectors |
| **Chroma** (client-server / cloud mode) | The same Chroma as above, run as a server |

There is also a third, very common group: **traditional databases that added vector search**. In practice they work as vector databases, and they are a good fit when your data already lives there:

- **PostgreSQL + pgvector**
- **Elasticsearch / OpenSearch**
- **MongoDB Atlas Vector Search**
- **Redis**

**Analogy:** a vector database is like a **public library**. It has a catalogue system, librarians, security, room for thousands of visitors, several branches, and safe copies in case one building has a problem.

## Side-by-side comparison

| Aspect | Vector Store | Vector Database |
|---|---|---|
| **What it is** | A library/component for storing and searching vectors | A full database system built around vectors |
| **Where it runs** | Inside your application's process | As a separate server or a managed cloud service |
| **Setup** | `pip install` and a few lines of code | Deploy a server or create a cloud account, then manage connections |
| **Persistence** | Often in memory; saving to a local file is usually up to you | Durable by default (disk storage, backups) |
| **Updates & deletes** | Basic; varies by library (some need the index rebuilt or re-saved) | Full CRUD, with changes searchable in near real time |
| **Metadata filtering** | Basic or none (often done by a wrapper after the search) | Built in, efficient, combined with the vector search |
| **Scale** | One machine, limited by its RAM | Scales out across machines to billions of vectors |
| **Concurrency** | One application / process | Many applications and users at once |
| **Security** | None built in | Authentication, roles, multi-tenancy, encryption |
| **Reliability** | A crash loses any unsaved data | Replication, backups, failover |
| **Cost** | Free (your own CPU and RAM) | Server or cloud cost (many have free tiers) |
| **Best for** | Learning, prototypes, notebooks, small or rarely changing datasets | Production apps, large or frequently changing data, many users |

## Is a vector database also a vector store?

Yes. Picture it as **nested circles**:

```
┌───────────────────────── Vector Store ─────────────────────────┐
│  "anything that can store vectors and run similarity search"   │
│                                                                │
│     FAISS, InMemoryVectorStore, Annoy, hnswlib, ...            │
│                                                                │
│   ┌──────────────────── Vector Database ───────────────────┐   │
│   │  "... plus persistence, CRUD, filtering, scaling,      │   │
│   │   concurrency, security and replication"               │   │
│   │                                                        │   │
│   │     Pinecone, Weaviate, Qdrant, Milvus, pgvector, ...  │   │
│   └────────────────────────────────────────────────────────┘   │
└────────────────────────────────────────────────────────────────┘
```

> **Every vector database is a vector store, but not every vector store is a vector database.**

### Why the terms get mixed up

- **Frameworks use one name for both.** In LangChain, `VectorStore` is the common interface for *every* backend: `FAISS`, `Chroma` and `PineconeVectorStore` all share the same methods (`add_documents`, `similarity_search`, `as_retriever`). So in LangChain code even Pinecone is "a vector store". That's useful: you can prototype with FAISS and later move to a vector database by changing only a few lines.
- **Some tools sit on both sides.** Chroma can run embedded in your notebook (a vector store) or as a server or cloud service (a vector database). Qdrant and Milvus also offer lightweight local modes (Qdrant local mode, Milvus Lite).
- **It's a spectrum, not a hard line.** The useful question isn't "what is this tool called?" but "which database features does my project need?"

## How the search stays fast: indexes (same in both)

Comparing a query against **every** stored vector (brute force) gives exact results, but it gets slow with millions of vectors. So vector stores and vector databases both use **Approximate Nearest Neighbour (ANN)** indexes, which give up a little accuracy for a big gain in speed.

| Index | The idea in one line | Trade-off |
|---|---|---|
| **Flat** (brute force) | Compare the query with every vector | 100% accurate, slow at large scale |
| **IVF** (Inverted File) | Group vectors into clusters and search only the closest clusters | Much faster; can miss results in clusters it skips |
| **HNSW** (Hierarchical Navigable Small World) | Build a layered graph of neighbours and "hop" towards the query | Very fast and accurate; uses more memory |
| **PQ** (Product Quantization) | Compress each vector into a short code | Large memory savings; small loss of accuracy |

FAISS exposes these indexes to you directly. A vector database uses the same kinds of algorithms (HNSW is the most common), but it builds, updates and distributes them across machines for you.

So **the search algorithm is not the difference**. The difference is the **database layer around it**.

## Where it fits in a RAG pipeline

```
INDEXING  (run once, and again whenever the documents change)

  documents ─► load & parse ─► chunk ─► embed ─┐
                                               ▼
                               ┌───────────────────────────────┐
                               │   Vector Store / Vector DB    │
                               │ (id, vector, text, metadata)  │
                               └───────────────────────────────┘
                                   ▲                       │
QUERYING  (every user question)    │ similarity search     │ top-k chunks
                                   │                       ▼
  user question ─► embed ──────────┘        prompt = question + chunks ─► LLM ─► answer
```

The vector store or database is the **bridge** between the two halves: indexing writes into it, and every question reads from it. Retrieval quality and speed depend heavily on it.

## Which one should I choose?

Go through these questions:

```
Is this a prototype, a notebook or a learning project?
 └─ yes ──► vector store (FAISS, Chroma local, InMemoryVectorStore)

Will the data outgrow the memory of one machine?
 └─ yes ──► vector database

Do documents change often (add / update / delete without rebuilding)?
 └─ yes ──► vector database

Do several users, services or servers need the same data at once?
 └─ yes ──► vector database

Do you need metadata filtering, access control or multi-tenancy?
 └─ yes ──► vector database

Does your data already live in Postgres / Elasticsearch / MongoDB?
 └─ yes ──► try its built-in vector search (pgvector, ...) before adding a new system
```

**A common path in practice**

1. **Learn and prototype** with an in-memory vector store (FAISS or Chroma): no infrastructure, instant feedback.
2. **Go to production** with a vector database (Pinecone, Qdrant, Weaviate, Milvus, pgvector, ...).
3. Because frameworks like LangChain share one `VectorStore` interface, step 2 usually means changing a few lines, not rewriting the app.

## Key takeaways

- A **vector store** keeps embeddings and finds the most similar ones. It is simple, local and usually in memory, which makes it ideal for learning and prototyping.
- A **vector database** runs the same search and adds **persistence, CRUD, metadata filtering, scaling, concurrency, security and reliability**, which is what production systems need.
- Both rely on the **same kinds of ANN indexes** (Flat, IVF, HNSW, PQ). The difference is the **database layer**, not the search maths.
- **Every vector database is a vector store, but not every vector store is a vector database.**
- The terms are used loosely: LangChain calls them all `VectorStore`, and tools like Chroma work both ways. Choose by the **features you need**, not by the label.

**Next:** we'll put this into practice by storing document chunks in a vector store and querying them.